# Phase 5, Notebook 05: Training Strategy & Loss Functions for VGGT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase5/05_training_strategy.ipynb)

## Learning Objectives

By the end of this notebook, you will:
- Understand the **CO3D dataset** and training data preparation
- Learn about **training configuration** (optimizer, learning rate, batch size)
- Implement **multi-task loss functions**: Camera Loss, Depth Loss, and Point Loss
- Explore **data augmentation strategies** used during training
- Understand **iterative refinement training**
- Visualize **training dynamics** and convergence patterns

## Estimated Time

**65 minutes**

## Prerequisites

- **Phase 5, Notebook 02**: Multi-task Prediction Heads
- Basic knowledge of PyTorch optimizers and loss functions

In [ ]:
# 环境设置 - Environment Setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# 设置随机种子以保证可重复性
np.random.seed(42)
torch.manual_seed(42)

print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Section 1: Training Data Overview (CO3D Dataset)

VGGT is trained on the **Common Objects in 3D (CO3D)** dataset, a large-scale collection of multi-view images of common objects.

### CO3D Dataset Statistics

| Property | Value |
|----------|-------|
| **Categories** | 50 object categories (apple, bench, car, etc.) |
| **Videos** | ~19,000 videos (captured by crowd-sourcing) |
| **Frames** | ~1.5 million frames |
| **Resolution** | Up to 1920×1080 (high quality) |
| **Views** | 15-60 frames per video |
| **GT Data** | Camera poses, depth maps, point clouds |

### Data Format

Each training sample consists of:
1. **Image sequence** `[S, 3, H, W]`: S frames from a video
2. **Camera poses** `[S, 9]`: (translation 3 + quaternion 4 + FoV 2) for each frame
3. **Depth maps** `[S, H, W]`: Per-frame depth maps with confidence
4. **Point maps** `[S, H, W, 3]`: 3D point clouds in camera space
5. **Tracks** `[S, N_q, 2]`: 2D correspondences for tracking (optional)

### Data Preprocessing

Before training, data undergoes several preprocessing steps:
1. **Frame sampling**: Select S=4-8 frames per video (spatially distributed)
2. **Resolution**: Resize to 518×518 or 224×224 depending on configuration
3. **Normalization**: ImageNet normalization for DINOv2 compatibility
4. **Depth scaling**: Normalize depth to [0.1, 100] meters range
5. **Camera normalization**: Center camera positions around origin

In [ ]:
# 模拟CO3D数据格式 - Simulating CO3D Data Format

class CO3DSample:
    """模拟CO3D数据集的一个样本"""
    
    def __init__(self, num_frames=4, height=518, width=518):
        self.num_frames = num_frames
        self.height = height
        self.width = width
        
    def generate_sample(self):
        """生成一个模拟的训练样本"""
        # 图像序列 [S, 3, H, W]
        images = torch.randn(self.num_frames, 3, self.height, self.width)
        
        # 相机位姿 [S, 9]: (trans[3] + quat[4] + fov[2])
        # 模拟围绕物体的相机轨迹
        poses = self._generate_camera_trajectory()
        
        # 深度图 [S, H, W]
        depths = torch.rand(self.num_frames, self.height, self.width) * 10.0 + 0.5
        
        # 深度置信度 [S, H, W]
        depth_confs = torch.rand(self.num_frames, self.height, self.width) + 1.0
        
        # 3D点图 [S, H, W, 3]
        point_maps = self._generate_point_maps(depths)
        
        # 置信度 [S, H, W]
        point_confs = torch.rand(self.num_frames, self.height, self.width) + 1.0
        
        return {
            'images': images,
            'poses': poses,
            'depths': depths,
            'depth_confs': depth_confs,
            'point_maps': point_maps,
            'point_confs': point_confs
        }
    
    def _generate_camera_trajectory(self):
        """生成围绕物体的相机轨迹"""
        poses = []
        
        # 圆形轨迹参数
        radius = 3.0
        height_offset = 0.5
        
        for i in range(self.num_frames):
            angle = 2 * np.pi * i / self.num_frames
            
            # 平移: 在圆周上
            trans = torch.tensor([
                radius * np.cos(angle),
                height_offset,
                radius * np.sin(angle)
            ])
            
            # 四元数: 指向原点 (简单起见,用单位四元数)
            quat = torch.tensor([1.0, 0.0, 0.0, 0.0])
            
            # FoV: 60度水平, 45度垂直
            fov = torch.tensor([60.0, 45.0])
            
            pose = torch.cat([trans, quat, fov])
            poses.append(pose)
        
        return torch.stack(poses)
    
    def _generate_point_maps(self, depths):
        """从深度图生成3D点图"""
        B, H, W = depths.shape
        
        # 创建像素网格
        u = torch.arange(W, dtype=torch.float32).view(1, 1, W).expand(B, H, W)
        v = torch.arange(H, dtype=torch.float32).view(1, H, 1).expand(B, H, W)
        
        # 假设相机内参
        fx = fy = 500.0
        cx = W / 2
        cy = H / 2
        
        # 反投影
        x = (u - cx) * depths / fx
        y = (v - cy) * depths / fy
        z = depths
        
        return torch.stack([x, y, z], dim=-1)


# 演示数据格式
print("=" * 60)
print("CO3D Dataset Sample Structure")
print("=" * 60)

sample_gen = CO3DSample(num_frames=4, height=224, width=224)
sample = sample_gen.generate_sample()

print("\n样本数据形状:")
for key, value in sample.items():
    print(f"  {key:15s}: {tuple(value.shape)}")

# 可视化相机轨迹
fig = plt.figure(figsize=(15, 5))

# 3D相机轨迹
ax1 = fig.add_subplot(131, projection='3d')
poses = sample['poses'].numpy()
ax1.plot(poses[:, 0], poses[:, 1], poses[:, 2], 'o-', markersize=10, linewidth=2)
ax1.plot(poses[0, 0], poses[0, 1], poses[0, 2], 'go', markersize=15, label='Start')
ax1.plot(poses[-1, 0], poses[-1, 1], poses[-1, 2], 'ro', markersize=15, label='End')
ax1.plot(0, 0, 0, 'k*', markersize=20, label='Object')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Camera Trajectory (3D)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.set_box_aspect([1,1,1])

# 深度图可视化
ax2 = fig.add_subplot(132)
depth_vis = sample['depths'][0].numpy()
im1 = ax2.imshow(depth_vis, cmap='viridis')
ax2.set_title('Depth Map (Frame 0)', fontsize=12, fontweight='bold')
ax2.axis('off')
plt.colorbar(im1, ax=ax2, fraction=0.046, pad=0.04)

# 置信度图
ax3 = fig.add_subplot(133)
conf_vis = sample['depth_confs'][0].numpy()
im2 = ax3.imshow(conf_vis, cmap='hot')
ax3.set_title('Depth Confidence (Frame 0)', fontsize=12, fontweight='bold')
ax3.axis('off')
plt.colorbar(im2, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("\nCO3D数据特点:")
print("  • 50个常见物体类别")
print("  • 相机围绕物体运动")
print("  • 提供相机位姿、深度、点云真值")
print("  • 支持多视图重建任务")

## Section 2: Training Configuration

VGGT uses a carefully designed training configuration to achieve optimal performance.

### Optimizer Configuration

```python
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-4,              # Initial learning rate
    betas=(0.9, 0.999),   # Beta coefficients
    weight_decay=0.05,    # Weight decay for regularization
)
```

**AdamW** is used instead of Adam because:
- Decouples weight decay from gradient updates
- Better generalization with L2 regularization
- Standard choice for Transformer-based models

### Learning Rate Schedule

VGGT uses a **warmup + cosine decay** schedule:

```
Phase 1: Linear warmup from 0 to max_lr (first 5% of steps)
Phase 2: Cosine decay from max_lr to min_lr (remaining 95% of steps)
```

```python
scheduler = OneCycleLR(
    optimizer=optimizer,
    max_lr=1e-4,
    total_steps=num_epochs * steps_per_epoch,
    pct_start=0.05,       # Warmup percentage
    anneal_strategy='cos',
)
```

### Batch Configuration

| Configuration | Value | Notes |
|---------------|-------|-------|
| **Batch size** | 32 | Per GPU (multi-GPU training) |
| **Frames (S)** | 4-8 | Number of views per sample |
| **Resolution** | 518×518 | High-res for quality |
| **Resolution** | 224×224 | Low-res for speed |
| **Epochs** | 100-200 | Depends on dataset size |

### Training Strategy

1. **Multi-scale training**: Alternate between 518×518 and 224×224
2. **Frame sampling**: Randomly sample 4-8 frames per video
3. **Gradient accumulation**: Effective batch size = 32 × num_gpus
4. **Mixed precision**: FP16/FP32 mixed training for efficiency
5. **Checkpointing**: Save best model based on validation loss

In [ ]:
# 训练配置可视化 - Training Configuration Visualization

class TrainingConfig:
    """VGGT训练配置类"""
    
    def __init__(self):
        # 优化器配置
        self.lr = 1e-4
        self.weight_decay = 0.05
        self.betas = (0.9, 0.999)
        
        # 调度器配置
        self.total_steps = 100000
        self.warmup_pct = 0.05
        self.min_lr = 1e-6
        
        # 数据配置
        self.batch_size = 32
        self.num_frames = 4
        self.resolution = 518  # or 224
        
        # 训练配置
        self.num_epochs = 100
        self.gradient_clip = 1.0


def compute_lr_schedule(step, total_steps, warmup_pct, max_lr, min_lr):
    """计算学习率调度"""
    warmup_steps = int(total_steps * warmup_pct)
    
    if step < warmup_steps:
        # Linear warmup
        return max_lr * (step / warmup_steps)
    else:
        # Cosine decay
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr + (max_lr - min_lr) * 0.5 * (1 + np.cos(np.pi * progress))


# 模拟训练配置
config = TrainingConfig()

print("=" * 60)
print("VGGT Training Configuration")
print("=" * 60)

print("\n优化器配置:")
print(f"  学习率: {config.lr}")
print(f"  权重衰减: {config.weight_decay}")
print(f"  Beta: {config.betas}")

print("\n调度器配置:")
print(f"  总步数: {config.total_steps}")
print(f"  Warmup比例: {config.warmup_pct * 100}%")
print(f"  最小学习率: {config.min_lr}")

print("\n数据配置:")
print(f"  批次大小: {config.batch_size}")
print(f"  每样本帧数: {config.num_frames}")
print(f"  分辨率: {config.resolution}×{config.resolution}")

# 可视化学习率调度
steps = np.arange(config.total_steps)
lr_schedule = [compute_lr_schedule(s, config.total_steps, config.warmup_pct, 
                                    config.lr, config.min_lr) for s in steps]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 学习率曲线
ax1 = axes[0]
ax1.plot(steps, lr_schedule, linewidth=2, color='blue')
ax1.axvline(x=config.total_steps * config.warmup_pct, color='r', linestyle='--', 
           label='End of Warmup')
ax1.set_xlabel('Training Step', fontsize=11)
ax1.set_ylabel('Learning Rate', fontsize=11)
ax1.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_yscale('log')

# 学习率的前5%(warmup)
ax2 = axes[1]
warmup_steps = int(config.total_steps * config.warmup_pct)
ax2.plot(steps[:warmup_steps], lr_schedule[:warmup_steps], linewidth=2, color='green')
ax2.set_xlabel('Training Step', fontsize=11)
ax2.set_ylabel('Learning Rate', fontsize=11)
ax2.set_title('Warmup Phase (5%)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n训练策略说明:")
print("  1. AdamW优化器: 解耦权重衰减")
print("  2. Warmup阶段: 防止早期梯度爆炸")
print("  3. Cosine衰减: 平滑收敛到最优解")
print("  4. 多分辨率训练: 平衡质量和速度")

## Section 3: Multi-task Loss Functions

VGGT uses a **weighted sum of multiple task-specific losses** for end-to-end training.

### Total Loss

```
L_total = λ_camera · L_camera + λ_depth · L_depth + λ_point · L_point
```

### Loss Weights

| Loss | Weight | Description |
|------|--------|-------------|
| **Camera Loss** | λ_camera = 1.0 | Camera pose and intrinsics |
| **Depth Loss** | λ_depth = 0.1 | Dense depth prediction |
| **Point Loss** | λ_point = 0.01 | 3D point cloud prediction |

### 3.1 Camera Loss (L_camera)

The Camera Loss supervises camera pose estimation with three components:

```python
L_camera = L_translation + L_rotation + L_fov
```

**Translation Loss (L1)**:
```python
L_translation = ||pred_trans - gt_trans||_1
```

**Rotation Loss (L1 on normalized quaternions)**:
```python
pred_quat_norm = pred_quat / ||pred_quat||
gt_quat_norm = gt_quat / ||gt_quat||
L_rotation = ||pred_quat_norm - gt_quat_norm||_1
```

**FoV Loss (L1)**:
```python
L_fov = ||pred_fov - gt_fov||_1
```

### 3.2 Depth Loss (L_depth)

The Depth Loss uses confidence-weighted L1 with gradient regularization:

```python
# Confidence-weighted L1
L_depth_l1 = mean(confidence * |pred_depth - gt_depth|) / mean(confidence)

# Gradient loss (edge preservation)
grad_pred = gradient(pred_depth)
grad_gt = gradient(gt_depth)
L_gradient = mean(|grad_pred - grad_gt|)

# Confidence regularization (encourage high confidence on accurate predictions)
L_conf_reg = -mean(confidence * exp(-|pred_depth - gt_depth|))

L_depth = L_depth_l1 + 0.5 * L_gradient + 0.1 * L_conf_reg
```

### 3.3 Point Loss (L_point)

The Point Loss supervises 3D point predictions:

```python
L_point = mean(confidence * ||pred_point - gt_point||_1) / mean(confidence)
```

In [ ]:
# 多任务损失函数实现 - Multi-task Loss Functions Implementation

class VGGTLoss(nn.Module):
    """VGGT多任务损失函数"""
    
    def __init__(self, lambda_camera=1.0, lambda_depth=0.1, lambda_point=0.01):
        super().__init__()
        self.lambda_camera = lambda_camera
        self.lambda_depth = lambda_depth
        self.lambda_point = lambda_point
    
    def camera_loss(self, pred_poses, gt_poses):
        """
        相机损失: L1损失用于平移、旋转和FoV
        
        Args:
            pred_poses: [B, S, 9] - 预测的位姿 (trans[3] + quat[4] + fov[2])
            gt_poses: [B, S, 9] - 真值位姿
        Returns:
            loss: scalar
        """
        # 分离各分量
        pred_trans = pred_poses[..., :3]
        pred_quat = pred_poses[..., 3:7]
        pred_fov = pred_poses[..., 7:9]
        
        gt_trans = gt_poses[..., :3]
        gt_quat = gt_poses[..., 3:7]
        gt_fov = gt_poses[..., 7:9]
        
        # 平移损失 (L1)
        trans_loss = F.l1_loss(pred_trans, gt_trans)
        
        # 旋转损失 (L1 on normalized quaternions)
        pred_quat_norm = F.normalize(pred_quat, dim=-1)
        gt_quat_norm = F.normalize(gt_quat, dim=-1)
        rotation_loss = F.l1_loss(pred_quat_norm, gt_quat_norm)
        
        # FoV损失 (L1)
        fov_loss = F.l1_loss(pred_fov, gt_fov)
        
        # 总相机损失
        total_camera_loss = trans_loss + rotation_loss + fov_loss
        
        return total_camera_loss, {
            'trans_loss': trans_loss.item(),
            'rotation_loss': rotation_loss.item(),
            'fov_loss': fov_loss.item()
        }
    
    def depth_loss(self, pred_depths, gt_depths, pred_confs, gt_confs):
        """
        深度损失: 置信度加权L1 + 梯度损失 + 置信度正则化
        
        Args:
            pred_depths: [B, S, H, W] - 预测深度
            gt_depths: [B, S, H, W] - 真值深度
            pred_confs: [B, S, H, W] - 预测置信度
            gt_confs: [B, S, H, W] - 真值置信度 (可选,用于mask)
        Returns:
            loss: scalar
        """
        # 置信度加权L1损失
        depth_diff = torch.abs(pred_depths - gt_depths)
        weighted_l1 = pred_confs * depth_diff
        l1_loss = weighted_l1.sum() / (pred_confs.sum() + 1e-8)
        
        # 梯度损失 (使用Sobel算子近似)
        def compute_gradient(x):
            """计算图像梯度"""
            # Sobel算子
            sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], 
                                  dtype=x.dtype, device=x.device).view(1, 1, 3, 3)
            sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], 
                                  dtype=x.dtype, device=x.device).view(1, 1, 3, 3)
            
            B, S, H, W = x.shape
            x_reshaped = x.view(B * S, 1, H, W)
            
            grad_x = F.conv2d(x_reshaped, sobel_x, padding=1)
            grad_y = F.conv2d(x_reshaped, sobel_y, padding=1)
            
            grad = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-8)
            return grad.view(B, S, H, W)
        
        grad_pred = compute_gradient(pred_depths)
        grad_gt = compute_gradient(gt_depths)
        gradient_loss = F.l1_loss(grad_pred, grad_gt)
        
        # 置信度正则化 (鼓励高置信度对应低误差)
        conf_reg = -(pred_confs * torch.exp(-depth_diff)).mean()
        
        # 总深度损失
        total_depth_loss = l1_loss + 0.5 * gradient_loss + 0.1 * conf_reg
        
        return total_depth_loss, {
            'depth_l1': l1_loss.item(),
            'depth_gradient': gradient_loss.item(),
            'depth_conf_reg': conf_reg.item()
        }
    
    def point_loss(self, pred_points, gt_points, pred_confs):
        """
        点云损失: 置信度加权L1
        
        Args:
            pred_points: [B, S, H, W, 3] - 预测3D点
            gt_points: [B, S, H, W, 3] - 真值3D点
            pred_confs: [B, S, H, W] - 预测置信度
        Returns:
            loss: scalar
        """
        # L1距离
        point_diff = torch.abs(pred_points - gt_points).sum(dim=-1)
        
        # 置信度加权
        weighted_diff = pred_confs * point_diff
        loss = weighted_diff.sum() / (pred_confs.sum() + 1e-8)
        
        return loss, {'point_l1': loss.item()}
    
    def forward(self, predictions, targets):
        """
        计算总损失
        
        Args:
            predictions: dict with keys ['poses', 'depths', 'depth_confs', 'points', 'point_confs']
            targets: dict with keys ['poses', 'depths', 'depth_confs', 'points', 'point_confs']
        Returns:
            total_loss: scalar
            loss_dict: dict of individual losses
        """
        losses = {}
        
        # 相机损失
        camera_loss, camera_details = self.camera_loss(
            predictions['poses'], targets['poses']
        )
        losses['camera'] = self.lambda_camera * camera_loss
        losses.update({f'camera_{k}': v for k, v in camera_details.items()})
        
        # 深度损失
        depth_loss, depth_details = self.depth_loss(
            predictions['depths'],
            targets['depths'],
            predictions['depth_confs'],
            targets['depth_confs']
        )
        losses['depth'] = self.lambda_depth * depth_loss
        losses.update({f'depth_{k}': v for k, v in depth_details.items()})
        
        # 点云损失
        point_loss, point_details = self.point_loss(
            predictions['points'],
            targets['points'],
            predictions['point_confs']
        )
        losses['point'] = self.lambda_point * point_loss
        losses.update({f'point_{k}': v for k, v in point_details.items()})
        
        # 总损失
        total_loss = losses['camera'] + losses['depth'] + losses['point']
        
        return total_loss, losses


# 测试损失函数
print("=" * 60)
print("Multi-task Loss Functions Demo")
print("=" * 60)

# 创建损失函数
criterion = VGGTLoss(lambda_camera=1.0, lambda_depth=0.1, lambda_point=0.01)

# 创建模拟预测和真值
B, S, H, W = 2, 4, 64, 64

predictions = {
    'poses': torch.randn(B, S, 9),
    'depths': torch.rand(B, S, H, W) * 10 + 0.5,
    'depth_confs': torch.rand(B, S, H, W) + 1.0,
    'points': torch.randn(B, S, H, W, 3),
    'point_confs': torch.rand(B, S, H, W) + 1.0
}

# 真值 (添加一些噪声)
targets = {
    'poses': predictions['poses'] + torch.randn(B, S, 9) * 0.1,
    'depths': predictions['depths'] + torch.randn(B, S, H, W) * 0.5,
    'depth_confs': predictions['depth_confs'],
    'points': predictions['points'] + torch.randn(B, S, H, W, 3) * 0.2,
    'point_confs': predictions['point_confs']
}

# 计算损失
total_loss, loss_dict = criterion(predictions, targets)

print(f"\n输入形状:")
print(f"  位姿: {predictions['poses'].shape}")
print(f"  深度: {predictions['depths'].shape}")
print(f"  点云: {predictions['points'].shape}")

print(f"\n损失值:")
print(f"  总损失: {total_loss.item():.4f}")
print(f"  ├── 相机损失 (λ={criterion.lambda_camera}): {losses['camera'].item():.4f}")
print(f"  │   ├── 平移: {loss_dict['camera_trans_loss']:.4f}")
print(f"  │   ├── 旋转: {loss_dict['camera_rotation_loss']:.4f}")
print(f"  │   └── FoV: {loss_dict['camera_fov_loss']:.4f}")
print(f"  ├── 深度损失 (λ={criterion.lambda_depth}): {losses['depth'].item():.4f}")
print(f"  │   ├── L1: {loss_dict['depth_depth_l1']:.4f}")
print(f"  │   ├── 梯度: {loss_dict['depth_depth_gradient']:.4f}")
print(f"  │   └── 置信度正则: {loss_dict['depth_depth_conf_reg']:.4f}")
print(f"  └── 点云损失 (λ={criterion.lambda_point}): {losses['point'].item():.4f}")
print(f"      └── L1: {loss_dict['point_point_l1']:.4f}")

# 可视化损失组成
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 加权损失饼图
ax1 = axes[0]
loss_values = [
    losses['camera'].item(),
    losses['depth'].item(), 
    losses['point'].item()
]
colors = ['#FFB6C1', '#FFD700', '#98FB98']
labels = ['Camera Loss', 'Depth Loss', 'Point Loss']
explode = (0.05, 0.05, 0.05)

ax1.pie(loss_values, labels=labels, colors=colors, autopct='%1.1f%%', 
        explode=explode, shadow=True, startangle=90)
ax1.set_title('Weighted Loss Composition', fontsize=12, fontweight='bold')

# 各损失组件的详细对比
ax2 = axes[1]
components = ['Trans', 'Rot', 'FoV', 'Depth L1', 'Grad', 'Conf Reg', 'Point']
values = [
    loss_dict['camera_trans_loss'],
    loss_dict['camera_rotation_loss'],
    loss_dict['camera_fov_loss'],
    loss_dict['depth_depth_l1'],
    loss_dict['depth_depth_gradient'],
    abs(loss_dict['depth_depth_conf_reg']),
    loss_dict['point_point_l1']
]
bar_colors = ['#FFB6C1', '#FFB6C1', '#FFB6C1', '#FFD700', '#FFD700', '#FFD700', '#98FB98']

bars = ax2.bar(components, values, color=bar_colors, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Loss Value', fontsize=11)
ax2.set_title('Individual Loss Components', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n损失函数设计要点:")
print("  1. 相机损失: 监督位姿的三个分量")
print("  2. 深度损失: 置信度加权 + 边缘保持")
print("  3. 点云损失: 直接监督3D位置")
print("  4. 损失权重: 平衡各任务的贡献")

## Section 4: Iterative Refinement Training

VGGT uses **iterative refinement** for both Camera Head and Track Head. This requires special training considerations.

### Iterative Refinement Loss

Instead of only supervising the final output, each iteration is supervised:

```python
# Camera Head produces outputs for 4 iterations
pose_predictions = [pose_iter1, pose_iter2, pose_iter3, pose_iter4]  # Each [B,S,9]

# Loss is computed for each iteration
L_camera_total = 0
for i, pred_pose in enumerate(pose_predictions):
    # Later iterations get higher weight
    weight = (i + 1) / 4  # [0.25, 0.5, 0.75, 1.0]
    L_camera_total += weight * L_camera(pred_pose, gt_pose)
```

### Progressive Supervision

| Iteration | Weight | Description |
|-----------|--------|-------------|
| **1** | 0.25 | Initial estimate |
| **2** | 0.50 | First refinement |
| **3** | 0.75 | Second refinement |
| **4** | 1.00 | Final output |

### Benefits

1. **Curriculum learning**: Easier tasks (early iterations) help learn harder tasks
2. **Better gradients**: Supervision at each step improves gradient flow
3. **Flexible inference**: Can stop at any iteration during inference for speed/accuracy trade-off

### Loss Scaling

To prevent early iterations from dominating:
- Use exponentially increasing weights: `weight = 2^i / sum(2^j)`
- Or use curriculum: start with uniform weights, increase later iterations over training

In [ ]:
# 迭代优化训练 - Iterative Refinement Training

class IterativeRefinementLoss(nn.Module):
    """
    迭代优化损失
    对每次迭代的输出都进行监督,后期迭代权重更高
    """
    
    def __init__(self, num_iterations=4, weighting='linear'):
        super().__init__()
        self.num_iterations = num_iterations
        self.weighting = weighting
        
        # 计算权重
        if weighting == 'linear':
            # 线性权重: [0.25, 0.5, 0.75, 1.0]
            weights = torch.arange(1, num_iterations + 1, dtype=torch.float32)
        elif weighting == 'exponential':
            # 指数权重
            weights = torch.exp2(torch.arange(num_iterations, dtype=torch.float32))
        else:  # uniform
            weights = torch.ones(num_iterations)
        
        self.register_buffer('weights', weights / weights.sum())
    
    def forward(self, predictions_list, target):
        """
        Args:
            predictions_list: List of [B, ...] with length num_iterations
            target: [B, ...] - ground truth
        Returns:
            total_loss: scalar
            per_iter_losses: list of losses for each iteration
        """
        total_loss = 0.0
        per_iter_losses = []
        
        for i, pred in enumerate(predictions_list):
            # 计算当前迭代的损失
            loss = F.l1_loss(pred, target)
            per_iter_losses.append(loss.item())
            
            # 加权求和
            total_loss += self.weights[i] * loss
        
        return total_loss, per_iter_losses


# 演示迭代优化损失
print("=" * 60)
print("Iterative Refinement Training Demo")
print("=" * 60)

# 创建模拟数据: 4次迭代的相机位姿预测
B, S = 2, 4
gt_poses = torch.randn(B, S, 9)

# 模拟每次迭代的预测 (逐渐接近真值)
predictions_iterative = []
for i in range(4):
    # 添加逐渐减少的噪声
    noise_scale = 1.0 / (i + 1)
    pred = gt_poses + torch.randn(B, S, 9) * noise_scale
    predictions_iterative.append(pred)

# 测试不同的权重策略
weighting_strategies = ['linear', 'exponential', 'uniform']
results = {}

for strategy in weighting_strategies:
    criterion = IterativeRefinementLoss(num_iterations=4, weighting=strategy)
    total_loss, per_iter_losses = criterion(predictions_iterative, gt_poses)
    results[strategy] = {
        'total': total_loss.item(),
        'per_iter': per_iter_losses,
        'weights': criterion.weights.numpy()
    }

print("\n不同权重策略的结果:")
for strategy, result in results.items():
    print(f"\n{strategy.capitalize()} Weighting:")
    print(f"  权重: {[f'{w:.3f}' for w in result['weights']]}")
    print(f"  每次迭代的损失: {[f'{l:.4f}' for l in result['per_iter']]}")
    print(f"  总损失: {result['total']:.4f}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, strategy in enumerate(weighting_strategies):
    ax = axes[idx]
    result = results[strategy]
    
    iterations = np.arange(1, 5)
    
    # 绘制损失曲线
    ax.plot(iterations, result['per_iter'], 'o-', linewidth=2, 
           markersize=8, label='Loss', color='blue')
    
    # 绘制权重柱状图
    ax2 = ax.twinx()
    ax2.bar(iterations, result['weights'], alpha=0.3, color='red', width=0.5, label='Weight')
    
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Loss Value', fontsize=11, color='blue')
    ax2.set_ylabel('Weight', fontsize=11, color='red')
    ax.set_title(f'{strategy.capitalize()} Weighting', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(iterations)
    ax.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # 添加图例
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

print("\n迭代优化训练要点:")
print("  1. 监督每次迭代的输出")
print("  2. 后期迭代有更高权重")
print("  3. 课程学习:从简单到复杂")
print("  4. 改善梯度传播")

## Section 5: Data Augmentation Strategies

Data augmentation is crucial for improving generalization and robustness.

### 5.1 Image Augmentations

| Augmentation | Parameters | Purpose |
|--------------|------------|---------|
| **Color Jitter** | brightness=0.2, contrast=0.2 | Robustness to lighting changes |
| **Random Horizontal Flip** | p=0.5 | Viewpoint invariance |
| **Random Rotation** | ±15 degrees | Rotation robustness |
| **Random Crop** | scale=[0.8, 1.0] | Scale invariance |
| **Gaussian Noise** | σ=0.01 | Robustness to sensor noise |

### 5.2 Geometric Augmentations

Geometric augmentations must be applied consistently to images and camera parameters:

```python
# 1. Random Horizontal Flip (requires quaternion update)
if random_flip:
    image = flip_horizontal(image)
    # Update camera: flip X coordinate
    camera_pose[0] *= -1  # translation X
    camera_quaternion = flip_quaternion_x(camera_quaternion)

# 2. Random Rotation (requires pose update)
if random_rotation:
    angle = random.uniform(-15, 15)
    image = rotate(image, angle)
    # Update camera rotation
    camera_quaternion = compose_rotations(camera_quaternion, angle_to_quat(angle))
```

### 5.3 Depth-aware Augmentations

```python
# 1. Depth noise (simulates sensor noise)
depth_noise = torch.randn_like(depth) * 0.05 * depth
depth = depth + depth_noise

# 2. Random depth scaling (scale augmentation)
scale = random.uniform(0.9, 1.1)
depth = depth * scale
camera_translation = camera_translation * scale

# 3. Occlusion simulation
if random_occlusion:
    mask = generate_random_mask(H, W)
    depth = depth * mask  # Set occluded regions to 0
    confidence = confidence * mask
```

### 5.4 Temporal Augmentations (for Videos)

```python
# 1. Random frame sampling
frame_indices = sorted(random.sample(range(total_frames), num_frames))

# 2. Temporal order shuffling (with probability 0.5)
if random_temporal_flip:
    frame_indices = frame_indices[::-1]

# 3. Frame dropout (simulate missing frames)
if random_dropout:
    dropout_mask = torch.rand(num_frames) > 0.1
    frame_indices = [f for f, m in zip(frame_indices, dropout_mask) if m]
```

In [ ]:
# 数据增强演示 - Data Augmentation Demo

class VGGTAugmentation:
    """VGGT数据增强类"""
    
    def __init__(self, image_size=224):
        self.image_size = image_size
    
    def color_jitter(self, image, brightness=0.2, contrast=0.2):
        """颜色抖动"""
        # 随机亮度调整
        brightness_factor = np.random.uniform(1 - brightness, 1 + brightness)
        image = image * brightness_factor
        
        # 随机对比度调整
        contrast_factor = np.random.uniform(1 - contrast, 1 + contrast)
        mean = image.mean(dim=(-2, -1), keepdim=True)
        image = (image - mean) * contrast_factor + mean
        
        return torch.clamp(image, 0, 1)
    
    def add_gaussian_noise(self, image, sigma=0.01):
        """添加高斯噪声"""
        noise = torch.randn_like(image) * sigma
        return image + noise
    
    def random_flip(self, image, depth, pose, p=0.5):
        """随机水平翻转"""
        if np.random.rand() < p:
            # 翻转图像和深度
            image = torch.flip(image, dims=[-1])
            depth = torch.flip(depth, dims=[-1])
            
            # 更新位姿 (翻转X坐标)
            pose[..., 0] *= -1  # translation X
            # 注意: 四元数也需要相应更新,这里简化处理
        
        return image, depth, pose
    
    def add_depth_noise(self, depth, sigma=0.05):
        """添加深度噪声"""
        noise = torch.randn_like(depth) * sigma * depth
        return torch.clamp(depth + noise, min=0.1)


# 生成模拟图像
def create_checkerboard(h, w, num_squares=8):
    """创建棋盘格图案"""
    y = np.linspace(-1, 1, h)
    x = np.linspace(-1, 1, w)
    xv, yv = np.meshgrid(x, y)
    pattern = ((xv * num_squares).astype(int) + (yv * num_squares).astype(int)) % 2
    return torch.tensor(pattern, dtype=torch.float32)


print("=" * 60)
print("Data Augmentation Strategies Demo")
print("=" * 60)

# 创建模拟数据
H, W = 128, 128
original_image = create_checkerboard(H, W, num_squares=8).unsqueeze(0)  # [1, H, W]
original_depth = torch.rand(H, W) * 5.0 + 1.0  # [H, W]
original_pose = torch.tensor([1.0, 0.5, 2.0, 1.0, 0.0, 0.0, 0.0, 60.0, 45.0])  # [9]

# 创建增强器
augmenter = VGGTAugmentation(image_size=H)

# 应用不同的增强
augmentations = {
    'Original': (original_image.clone(), original_depth.clone(), original_pose.clone()),
    'Color Jitter': (
        augmenter.color_jitter(original_image.clone(), brightness=0.3, contrast=0.3),
        original_depth.clone(),
        original_pose.clone()
    ),
    'Gaussian Noise': (
        augmenter.add_gaussian_noise(original_image.clone(), sigma=0.05),
        original_depth.clone(),
        original_pose.clone()
    ),
    'Depth Noise': (
        original_image.clone(),
        augmenter.add_depth_noise(original_depth.clone(), sigma=0.1),
        original_pose.clone()
    ),
}

# 可视化
fig, axes = plt.subplots(len(augmentations), 2, figsize=(10, len(augmentations) * 3))

for idx, (name, (img, depth, pose)) in enumerate(augmentations.items()):
    # 显示图像
    ax_img = axes[idx, 0]
    ax_img.imshow(img[0].numpy(), cmap='gray')
    ax_img.set_title(f'{name} - Image', fontsize=11, fontweight='bold')
    ax_img.axis('off')
    
    # 显示深度
    ax_depth = axes[idx, 1]
    im = ax_depth.imshow(depth.numpy(), cmap='viridis')
    ax_depth.set_title(f'{name} - Depth', fontsize=11, fontweight='bold')
    ax_depth.axis('off')
    plt.colorbar(im, ax=ax_depth, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 可视化增强策略概览
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# 增强策略列表
augmentation_list = [
    {'name': 'Color Jitter', 'category': 'Image', 'y': 7, 'x': 1, 'color': '#FFB6C1'},
    {'name': 'Gaussian Noise', 'category': 'Image', 'y': 6, 'x': 1, 'color': '#FFB6C1'},
    {'name': 'Random Flip', 'category': 'Geometric', 'y': 5, 'x': 1, 'color': '#FFD700'},
    {'name': 'Random Crop', 'category': 'Geometric', 'y': 4, 'x': 1, 'color': '#FFD700'},
    {'name': 'Depth Noise', 'category': 'Depth-aware', 'y': 3, 'x': 1, 'color': '#98FB98'},
    {'name': 'Depth Scaling', 'category': 'Depth-aware', 'y': 2, 'x': 1, 'color': '#98FB98'},
    {'name': 'Occlusion', 'category': 'Depth-aware', 'y': 1, 'x': 1, 'color': '#98FB98'},
    {'name': 'Frame Sampling', 'category': 'Temporal', 'y': 0, 'x': 1, 'color': '#DDA0DD'},
]

# 类别标题
categories = {
    'Image': (0, 7, '#FFB6C1'),
    'Geometric': (3, 5, '#FFD700'),
    'Depth-aware': (5, 3, '#98FB98'),
    'Temporal': (8, 1, '#DDA0DD')
}

# 绘制类别区域
for cat, (x, y_range, color) in categories.items():
    rect = patches.Rectangle((x, y_range - 0.5), 3, 1, 
                             linewidth=2, edgecolor='black', 
                             facecolor=color, alpha=0.3)
    ax.add_patch(rect)
    ax.text(x + 1.5, y_range, cat, ha='center', va='center', 
           fontsize=12, fontweight='bold')

# 绘制增强项
for aug in augmentation_list:
    rect = FancyBboxPatch((aug['x'] + 4, aug['y'] - 0.3), 2, 0.6,
                         boxstyle="round,pad=0.05",
                         edgecolor='black', facecolor=aug['color'],
                         alpha=0.7, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(aug['x'] + 5, aug['y'], aug['name'], ha='center', va='center',
           fontsize=10, fontweight='bold')

ax.set_title('VGGT Data Augmentation Strategies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n数据增强策略总结:")
print("  图像增强: 颜色抖动、高斯噪声")
print("  几何增强: 水平翻转、随机裁剪 (需要更新相机参数)")
print("  深度感知: 深度噪声、缩放、遮挡模拟")
print("  时序增强: 随机帧采样、时序顺序打乱")

## Section 6: Training Dynamics Visualization

Understanding training dynamics helps diagnose issues and optimize hyperparameters.

### Key Metrics to Monitor

1. **Total Loss** - Overall training progress
2. **Task-specific Losses** - Individual task performance
3. **Learning Rate** - Schedule verification
4. **Gradient Norm** - Training stability
5. **Validation Metrics** - Generalization

### Loss Convergence Patterns

```
Healthy Training:
- Total loss decreases smoothly
- Task losses balanced (similar magnitude)
- Validation loss follows training loss

Unhealthy Training:
- Loss oscillates wildly → Reduce learning rate
- Loss plateaus early → Increase learning rate or check data
- Validation loss diverges → Overfitting, add regularization
- One task dominates → Adjust loss weights
```

### Gradient Flow

Monitoring gradient norms helps detect:
- **Vanishing gradients**: Norm < 1e-6
- **Exploding gradients**: Norm > 100
- **Gradient clipping**: Prevents exploding gradients

In [ ]:
# 训练动态可视化 - Training Dynamics Visualization

def simulate_training(num_steps=1000, num_tasks=3):
    """
    模拟训练过程
    返回损失历史和学习率历史
    """
    # 模拟损失曲线 (带噪声的指数衰减)
    t = np.linspace(0, 1, num_steps)
    
    # 相机损失 (快速收敛)
    camera_loss = 2.0 * np.exp(-3 * t) + 0.1 + np.random.normal(0, 0.05, num_steps)
    
    # 深度损失 (较慢收敛)
    depth_loss = 5.0 * np.exp(-2 * t) + 0.5 + np.random.normal(0, 0.1, num_steps)
    
    # 点云损失 (最慢收敛)
    point_loss = 10.0 * np.exp(-1.5 * t) + 1.0 + np.random.normal(0, 0.2, num_steps)
    
    # 总损失 (加权组合)
    total_loss = (1.0 * camera_loss + 
                  0.1 * depth_loss + 
                  0.01 * point_loss)
    
    # 学习率调度 (warmup + cosine decay)
    warmup_steps = int(0.05 * num_steps)
    lr = np.zeros(num_steps)
    
    for i in range(num_steps):
        if i < warmup_steps:
            lr[i] = 1e-4 * (i / warmup_steps)
        else:
            progress = (i - warmup_steps) / (num_steps - warmup_steps)
            lr[i] = 1e-6 + (1e-4 - 1e-6) * 0.5 * (1 + np.cos(np.pi * progress))
    
    # 梯度范数 (模拟)
    grad_norm = 10.0 * np.exp(-t) + 1.0 + np.random.normal(0, 0.5, num_steps)
    grad_norm = np.maximum(grad_norm, 0.1)  # 最小值
    
    return {
        'camera_loss': camera_loss,
        'depth_loss': depth_loss,
        'point_loss': point_loss,
        'total_loss': total_loss,
        'learning_rate': lr,
        'grad_norm': grad_norm
    }


# 生成训练历史
print("=" * 60)
print("Training Dynamics Visualization")
print("=" * 60)

history = simulate_training(num_steps=1000)

# 创建子图
fig = plt.figure(figsize=(16, 10))

# 1. 损失曲线
ax1 = plt.subplot(2, 3, 1)
steps = np.arange(len(history['total_loss']))
ax1.plot(steps, history['total_loss'], label='Total Loss', linewidth=2, color='blue')
ax1.plot(steps, history['camera_loss'], label='Camera Loss', linewidth=1.5, alpha=0.7)
ax1.plot(steps, history['depth_loss'], label='Depth Loss', linewidth=1.5, alpha=0.7)
ax1.plot(steps, history['point_loss'], label='Point Loss', linewidth=1.5, alpha=0.7)
ax1.set_xlabel('Training Step', fontsize=11)
ax1.set_ylabel('Loss Value', fontsize=11)
ax1.set_title('Training Loss Curves', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 2. 学习率曲线
ax2 = plt.subplot(2, 3, 2)
ax2.plot(steps, history['learning_rate'], linewidth=2, color='green')
ax2.set_xlabel('Training Step', fontsize=11)
ax2.set_ylabel('Learning Rate', fontsize=11)
ax2.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# 3. 梯度范数
ax3 = plt.subplot(2, 3, 3)
ax3.plot(steps, history['grad_norm'], linewidth=2, color='red')
ax3.axhline(y=1.0, color='orange', linestyle='--', label='Typical threshold')
ax3.set_xlabel('Training Step', fontsize=11)
ax3.set_ylabel('Gradient Norm', fontsize=11)
ax3.set_title('Gradient Norm', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 损失组成 (堆叠面积图)
ax4 = plt.subplot(2, 3, 4)
ax4.stackplot(steps, 
              history['camera_loss'],
              history['depth_loss'] * 0.1,
              history['point_loss'] * 0.01,
              labels=['Camera', 'Depth (×0.1)', 'Point (×0.01)'],
              colors=['#FFB6C1', '#FFD700', '#98FB98'],
              alpha=0.7)
ax4.set_xlabel('Training Step', fontsize=11)
ax4.set_ylabel('Weighted Loss', fontsize=11)
ax4.set_title('Loss Composition Over Time', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3)

# 5. 损失下降速率
ax5 = plt.subplot(2, 3, 5)
window = 50
loss_smooth = np.convolve(history['total_loss'], np.ones(window)/window, mode='valid')
loss_derivative = -np.diff(loss_smooth)  # 负导数表示下降
ax5.plot(steps[window:-1], loss_derivative, linewidth=2, color='purple')
ax5.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax5.set_xlabel('Training Step', fontsize=11)
ax5.set_ylabel('Loss Decrease Rate', fontsize=11)
ax5.set_title('Loss Decrease Rate (Smoothed)', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6. 收敛分析
ax6 = plt.subplot(2, 3, 6)
# 计算移动平均
ma_window = 100
camera_ma = np.convolve(history['camera_loss'], np.ones(ma_window)/ma_window, mode='valid')
depth_ma = np.convolve(history['depth_loss'], np.ones(ma_window)/ma_window, mode='valid')
point_ma = np.convolve(history['point_loss'], np.ones(ma_window)/ma_window, mode='valid')

ax6.plot(steps[ma_window-1:], camera_ma, label='Camera (MA)', linewidth=2)
ax6.plot(steps[ma_window-1:], depth_ma, label='Depth (MA)', linewidth=2)
ax6.plot(steps[ma_window-1:], point_ma, label='Point (MA)', linewidth=2)
ax6.set_xlabel('Training Step', fontsize=11)
ax6.set_ylabel('Loss Value (Moving Avg)', fontsize=11)
ax6.set_title('Smoothed Loss Curves', fontsize=12, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.set_yscale('log')

plt.tight_layout()
plt.show()

print("\n训练动态分析:")
print(f"  初始总损失: {history['total_loss'][0]:.4f}")
print(f"  最终总损失: {history['total_loss'][-1]:.4f}")
print(f"  损失下降: {(1 - history['total_loss'][-1]/history['total_loss'][0])*100:.1f}%")
print("\n各任务收敛速度:")
print(f"  相机损失: 快 (快速学习位姿)")
print(f"  深度损失: 中等 (空间细节需要更多时间)")
print(f"  点云损失: 慢 (3D几何最难学习)")
print("\n健康指标:")
print(f"  梯度范数范围: [{history['grad_norm'].min():.2f}, {history['grad_norm'].max():.2f}]")
print(f"  学习率范围: [{history['learning_rate'].min():.2e}, {history['learning_rate'].max():.2e}]")

## Section 7: Complete Training Loop Example

Putting it all together, here's a complete training loop example:

In [ ]:
# 完整的训练循环示例 - Complete Training Loop Example

def training_loop_demo():
    """
    完整的VGGT训练循环示例
    (演示用,非可运行代码)
    """
    
    code_example = '''
# 1. 设置
model = VGGT()
criterion = VGGTLoss(lambda_camera=1.0, lambda_depth=0.1, lambda_point=0.01)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = OneCycleLR(optimizer, max_lr=1e-4, total_steps=total_steps, pct_start=0.05)

# 2. 训练循环
for epoch in range(num_epochs):
    model.train()
    
    for batch_idx, batch in enumerate(train_loader):
        # 2.1 数据准备
        images = batch['images'].to(device)  # [B, S, 3, H, W]
        gt_poses = batch['poses'].to(device)  # [B, S, 9]
        gt_depths = batch['depths'].to(device)  # [B, S, H, W]
        gt_points = batch['points'].to(device)  # [B, S, H, W, 3]
        
        # 2.2 前向传播
        predictions = model(images)
        # predictions包含多任务输出
        
        # 2.3 计算损失
        total_loss, loss_dict = criterion(predictions, batch)
        
        # 2.4 反向传播
        optimizer.zero_grad()
        total_loss.backward()
        
        # 2.5 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 2.6 优化器步进
        optimizer.step()
        scheduler.step()
        
        # 2.7 日志记录
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}")
            print(f"  Loss: {total_loss.item():.4f}")
            print(f"    - Camera: {loss_dict['camera']:.4f}")
            print(f"    - Depth: {loss_dict['depth']:.4f}")
            print(f"    - Point: {loss_dict['point']:.4f}")
    
    # 3. 验证
    model.eval()
    val_loss = validate(model, val_loader, criterion)
    
    # 4. 保存检查点
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
    '''
    
    return code_example


print("=" * 60)
print("Complete Training Loop Example")
print("=" * 60)
print(training_loop_demo())

# 训练检查清单
print("\n" + "=" * 60)
print("Training Checklist")
print("=" * 60)

checklist = [
    "✓ 数据加载: CO3D数据预处理正确",
    "✓ 数据增强: 图像和几何增强实现",
    "✓ 损失函数: 多任务损失加权正确",
    "✓ 优化器: AdamW配置",
    "✓ 学习率调度: Warmup + Cosine Decay",
    "✓ 迭代优化: 监督所有迭代输出",
    "✓ 梯度裁剪: 防止梯度爆炸",
    "✓ 混合精度: FP16/FP32训练",
    "✓ 验证循环: 定期评估",
    "✓ 检查点: 保存最佳模型",
]

for item in checklist:
    print(f"  {item}")

print("\n训练调优技巧:")
print("  1. 从小的学习率开始,观察损失趋势")
print("  2. 监控各任务的损失比例,调整权重")
print("  3. 使用TensorBoard可视化训练过程")
print("  4. 定期保存检查点,防止训练中断")
print("  5. 验证集上早停,防止过拟合")

## Summary

```
╔═══════════════════════════════════════════════════════════════════════╗
║                  Key Takeaways - Training Strategy                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  1. Training Data (CO3D Dataset):                                    ║
║     • 50 object categories, ~19K videos, ~1.5M frames                ║
║     • Provides GT: poses, depth, point clouds                        ║
║     • Multi-view consistent annotations                              ║
║                                                                       ║
║  2. Training Configuration:                                          ║
║     • Optimizer: AdamW (lr=1e-4, weight_decay=0.05)                  ║
║     • Scheduler: Warmup + Cosine Decay                               ║
║     • Batch size: 32, Frames: 4-8, Resolution: 518×518 or 224×224    ║
║                                                                       ║
║  3. Multi-task Loss Functions:                                       ║
║     • Camera Loss: L1 for translation, rotation (quaternion), FoV    ║
║     • Depth Loss: Confidence-weighted L1 + gradient + conf reg       ║
║     • Point Loss: Confidence-weighted L1 for 3D points               ║
║     • Weights: λ_camera=1.0, λ_depth=0.1, λ_point=0.01               ║
║                                                                       ║
║  4. Data Augmentation Strategies:                                    ║
║     • Image: Color jitter, Gaussian noise                            ║
║     • Geometric: Horizontal flip, random crop (update camera)        ║
║     • Depth-aware: Depth noise, scaling, occlusion                   ║
║     • Temporal: Random frame sampling, order shuffling               ║
║                                                                       ║
║  5. Iterative Refinement Training:                                   ║
║     • Supervise all 4 iterations with increasing weights             ║
║     • Linear or exponential weighting strategies                     ║
║     • Curriculum learning from easy to hard                          ║
║                                                                       ║
║  6. Training Dynamics Monitoring:                                    ║
║     • Monitor total loss, task losses, learning rate                 ║
║     • Track gradient norms for stability                             ║
║     • Balance loss magnitudes across tasks                           ║
║                                                                       ║
║  7. Best Practices:                                                  ║
║     • Use mixed precision training (FP16)                            ║
║     • Apply gradient clipping (max_norm=1.0)                         ║
║     • Save checkpoints regularly                                     ║
║     • Validate on held-out data                                      ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

## What's Next?

In the next notebook, we'll explore:

**Phase 5, Notebook 06: Code Walkthrough**
- Complete VGGT model implementation
- Inference pipeline
- Post-processing and output generation

Continue to [06_code_walkthrough.ipynb](./06_code_walkthrough.ipynb)

---

## References

1. **VGGT**: [Video Gaussian Gaussian Transformer](https://arxiv.org/abs/TODO)
2. **CO3D Dataset**: Reizenstein et al., "Common Objects in 3D", CVPR 2021
3. **AdamW**: Loshchilov & Hutter, "Decoupled Weight Decay Regularization", ICLR 2019
4. **OneCycleLR**: Smith & Topin, "Super-Convergence", 2019
5. **Multi-task Learning**: Kendall et al., "Multi-Task Learning Using Uncertainty", CVPR 2018